# 04 - Loading the data into MySQL and creating reporting views

So far, every clean table has lived only as a CSV in
`datasets/processed/`. In this notebook I load them into MySQL, so
Power BI can read from one organized place instead of a bunch of
loose CSV files.

Why use a database? Because with SQL I can **join** tables and run
queries that a single CSV can't answer -- for example, linking
production to quality through the same work order.


In [ ]:
import os
import glob
import pandas as pd
from sqlalchemy import create_engine, text

PROCESSED = "../../datasets/processed"
DIM = "../../datasets/dim"
SQL_TABLE = "../sql_table"
SQL_VIEW = "../sql_view"


## Step 1: connection details

I read the username and password from **environment variables** (or a
`.env` file), never hard-coded in the notebook -- if this notebook goes
on GitHub, the password shouldn't go with it.

Before running this notebook, set the following (in a terminal, or in
a `.env` file at the project root):

```
MYSQL_HOST=localhost
MYSQL_PORT=3306
MYSQL_USER=root
MYSQL_PASSWORD=your_password_here
MYSQL_DB=manufacturing_performance_analytics
```


In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv("../../.env")
except ImportError:
    print("python-dotenv is not installed -- run: pip install python-dotenv")

user = os.environ.get("MYSQL_USER", "root")
password = os.environ.get("MYSQL_PASSWORD", "")
host = os.environ.get("MYSQL_HOST", "localhost")
port = os.environ.get("MYSQL_PORT", "3306")
database = os.environ.get("MYSQL_DB", "manufacturing_performance_analytics")

print(f"Connecting to: {user}@{host}:{port}/{database}")


## Step 2: create the database (if it doesn't exist yet)

In [ ]:
# connect to the server WITHOUT specifying a database yet (it may not exist)
server_connection = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}")

with server_connection.begin() as connection:
    connection.execute(text(f"CREATE DATABASE IF NOT EXISTS {database} CHARACTER SET utf8mb4"))

print(f"Database '{database}' ready.")

# now a connection pointed straight at the database
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}?charset=utf8mb4")


## Step 3: create the tables (schema)

The `01_create_fact_tables.sql` and `02_create_dim_tables.sql` files
already have the `CREATE TABLE` statement for each table, with the
right column types. A simple function reads the `.sql` file, splits it
on `;` (each statement ends with a semicolon), and runs each one.


In [ ]:
def run_sql_file(engine, file_path):
    with open(file_path, encoding="utf-8") as f:
        full_sql = f.read()

    # every statement ends with ; -- I split on that, and skip comment
    # lines (which start with --) and blank lines
    statements = [s.strip() for s in full_sql.split(";") if s.strip() and not s.strip().startswith("--")]

    with engine.begin() as connection:
        for statement in statements:
            connection.execute(text(statement))

    print(f"{len(statements)} statements executed from {file_path}")


run_sql_file(engine, f"{SQL_TABLE}/01_create_fact_tables.sql")
run_sql_file(engine, f"{SQL_TABLE}/02_create_dim_tables.sql")


## Step 4: load the data

Every CSV file in `datasets/processed/` becomes a table with the same
name (minus the `.csv`). I use that rule so I'm not typing out the
name of each of the 34 tables by hand -- I just loop over the files in
the folder.

Before loading, I empty the table first (`TRUNCATE`), so this notebook
can be run again without duplicating any rows.


In [ ]:
def load_csv_into_table(engine, csv_path, table_name):
    df = pd.read_csv(csv_path, encoding="utf-8-sig")

    with engine.begin() as connection:
        connection.execute(text(f"TRUNCATE TABLE {table_name}"))

    # chunksize=500: sends 500 rows at a time, instead of everything at
    # once -- avoids going over MySQL's message-size limit
    df.to_sql(table_name, engine, if_exists="append", index=False, chunksize=500, method="multi")

    print(f"loaded: {table_name} ({len(df):,} rows)")


# 1) every fact/dimension table already sitting in datasets/processed
for path in sorted(glob.glob(f"{PROCESSED}/*.csv")):
    table_name = os.path.basename(path).replace(".csv", "")
    load_csv_into_table(engine, path, table_name)

# 2) the dimension tables that come straight from datasets/dim (these
#    don't go through the cleaning notebooks, they arrive ready from engineering)
dim_files_loaded_directly = [
    "dim_masterbatch.csv",
    "dim_machine_setup.csv",
    "dim_cap_control_plan_cq.csv",
    "dim_bottle_control_plan_cq.csv",
    "dim_ink_control_plan_cq.csv",
]
for file_name in dim_files_loaded_directly:
    path = f"{DIM}/{file_name}"
    table_name = file_name.replace(".csv", "")
    load_csv_into_table(engine, path, table_name)


## Step 5: create the views (rolling 52-week window)

A **view** is a saved SQL query that behaves like a table. The four
views here always filter to the most recent 52 weeks of data -- that's
what Power BI uses, not the full tables (otherwise the dashboard would
show the entire accumulated history, with no rolling window).


In [ ]:
view_files = [
    "01_production_by_process_52w.sql",
    "02_downtime_material_plan_52w.sql",
    "03_quality_control_52w.sql",
    "04_quality_assurance_52w.sql",
    "05_dim_views.sql",
]

for file_name in view_files:
    run_sql_file(engine, f"{SQL_VIEW}/{file_name}")

print("Views created. Power BI can now connect (tables starting with vw_).")
